# Лабораторная работа №4: Проведение исследований со случайным лесом

## Цель работы
Исследование алгоритмов случайного леса для задач классификации и регрессии на реальных данных. Работа включает создание бейзлайна с использованием библиотеки sklearn, его улучшение и самостоятельную имплементацию алгоритмов.

## Используемые датасеты
- **Классификация:** "Human Activity Recognition with Smartphones" — данные с акселерометра и гироскопа смартфона для определения 6 видов активности человека.
- **Регрессия:** "CO2 Emission by Vehicles" — характеристики автомобилей и уровень выбросов CO₂.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Загрузка данных для классификации
train_class = pd.read_csv('datasets/HUMAN_ACTIVITY/train.csv')
test_class = pd.read_csv('datasets/HUMAN_ACTIVITY/test.csv')

# Загрузка данных для регрессии
data_reg = pd.read_csv('datasets/CO2/CO2_dataset.csv')

print("Данные классификации загружены:")
print(f"  Train: {train_class.shape}, Test: {test_class.shape}")
print(f"\nДанные регрессии загружены: {data_reg.shape}")

Данные классификации загружены:
  Train: (7352, 563), Test: (2947, 563)

Данные регрессии загружены: (7385, 12)


## Предварительная обработка данных

Первичный анализ данных был проведен в первой ЛР, поэтому здесь сразу переходим к подготовке данных

### Для классификации:
1. Разделение на признаки и целевую переменную
2. Кодирование категориальной целечной переменной в числовой формат

### Для регрессии:
1. Выбор числовых признаков и целевой переменной
2. Разделение на тренировочную и тестовую выборки

Для случайного леса масштабирование признаков не требуется.

In [3]:
# Подготовка данных для классификации
X_train_class = train_class.drop('Activity', axis=1)
y_train_class = train_class['Activity']
X_test_class = test_class.drop('Activity', axis=1)
y_test_class = test_class['Activity']

# Кодирование меток классов
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_class)
y_test_encoded = le.transform(y_test_class)

# Подготовка данных для регрессии
numeric_cols = data_reg.select_dtypes(include=[np.number]).columns
X_reg = data_reg[numeric_cols].drop('CO2 Emissions(g/km)', axis=1)
y_reg = data_reg['CO2 Emissions(g/km)']

# Разделение на train/test
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

print("Данные успешно подготовлены:")
print(f"Классификация: X_train {X_train_class.shape}, y_train {y_train_encoded.shape}")
print(f"Регрессия: X_train {X_train_reg.shape}, y_train {y_train_reg.shape}")

Данные успешно подготовлены:
Классификация: X_train (7352, 562), y_train (7352,)
Регрессия: X_train (5908, 6), y_train (5908,)


## Функции для вычисления метрик качества

Для задач классификации используем Accuracy, Precision, Recall, F1-Score и ROC-AUC. Для регрессии используем MSE, MAE и R².

In [4]:
# Функции для вычисления метрик
def classification_metrics(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}
    if y_prob is not None:
        try:
            auc = roc_auc_score(y_true, y_prob, multi_class='ovr')
            metrics['ROC-AUC'] = auc
        except:
            metrics['ROC-AUC'] = None
    return metrics

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'MSE': mse, 'MAE': mae, 'R2': r2}

## Бейзлайн: классификация с использованием RandomForestClassifier из sklearn

In [5]:
# Инициализация и обучение модели случайного леса для классификации
rf_class = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_class.fit(X_train_class, y_train_encoded)

# Прогнозы
y_pred_rf_class = rf_class.predict(X_test_class)
y_prob_rf_class = rf_class.predict_proba(X_test_class)

# Оценка метрик
metrics_rf_class = classification_metrics(y_test_encoded, y_pred_rf_class, y_prob_rf_class)
print("Метрики классификации (случайный лес, бейзлайн):")
for key, value in metrics_rf_class.items():
    print(f"{key}: {value:.4f}")

print(f"\nКоличество деревьев: {rf_class.n_estimators}")
print(f"Глубина деревьев (средняя): {np.mean([tree.get_depth() for tree in rf_class.estimators_]):.2f}")

Метрики классификации (случайный лес, бейзлайн):
Accuracy: 0.9267
Precision: 0.9281
Recall: 0.9237
F1-Score: 0.9250
ROC-AUC: 0.9949

Количество деревьев: 100
Глубина деревьев (средняя): 18.62


## Бейзлайн: регрессия с использованием RandomForestRegressor из sklearn

In [6]:
# Инициализация и обучение модели случайного леса для регрессии
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_reg, y_train_reg)

# Прогнозы
y_pred_rf_reg = rf_reg.predict(X_test_reg)

# Оценка метрик
metrics_rf_reg = regression_metrics(y_test_reg, y_pred_rf_reg)
print("Метрики регрессии (случайный лес, бейзлайн):")
for key, value in metrics_rf_reg.items():
    print(f"{key}: {value:.4f}")

print(f"\nКоличество деревьев: {rf_reg.n_estimators}")
print(f"Глубина деревьев (средняя): {np.mean([tree.get_depth() for tree in rf_reg.estimators_]):.2f}")

Метрики регрессии (случайный лес, бейзлайн):
MSE: 84.9714
MAE: 3.2122
R2: 0.9753

Количество деревьев: 100
Глубина деревьев (средняя): 21.22


## Улучшение бейзлайна: гипотезы

Для случайного леса важными гиперпараметрами являются:
1. Количество деревьев (n_estimators)
2. Максимальная глубина деревьев (max_depth)
3. Минимальное количество образцов для разделения узла (min_samples_split)
4. Минимальное количество образцов в листе (min_samples_leaf)
5. Количество признаков для рассмотрения при каждом разделении (max_features)

Используем GridSearchCV для подбора оптимальных гиперпараметров.

In [7]:
# Подбор гиперпараметров для случайного леса классификации
param_grid_rf_class = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

grid_rf_class = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), 
                             param_grid_rf_class, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_rf_class.fit(X_train_class, y_train_encoded)

print("Лучшие параметры для случайного леса (классификация):", grid_rf_class.best_params_)
print("Лучшая accuracy на кросс-валидации:", grid_rf_class.best_score_)

# Подбор гиперпараметров для случайного леса регрессии
param_grid_rf_reg = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

grid_rf_reg = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), 
                           param_grid_rf_reg, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)
grid_rf_reg.fit(X_train_reg, y_train_reg)

print("\nЛучшие параметры для случайного леса (регрессия):", grid_rf_reg.best_params_)
print("Лучший MSE на кросс-валидации:", -grid_rf_reg.best_score_)

Fitting 3 folds for each of 216 candidates, totalling 648 fits
Лучшие параметры для случайного леса (классификация): {'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Лучшая accuracy на кросс-валидации: 0.9265518724829792
Fitting 3 folds for each of 216 candidates, totalling 648 fits

Лучшие параметры для случайного леса (регрессия): {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Лучший MSE на кросс-валидации: 56.06766718140051


## Формирование улучшенного бейзлайна

In [8]:
# Улучшенная модель случайного леса для классификации
best_rf_class = grid_rf_class.best_estimator_
y_pred_rf_class_improved = best_rf_class.predict(X_test_class)
y_prob_rf_class_improved = best_rf_class.predict_proba(X_test_class)

metrics_rf_class_improved = classification_metrics(y_test_encoded, y_pred_rf_class_improved, y_prob_rf_class_improved)
print("Метрики классификации (улучшенный случайный лес):")
for key, value in metrics_rf_class_improved.items():
    print(f"{key}: {value:.4f}")

print(f"\nКоличество деревьев: {best_rf_class.n_estimators}")
print(f"Максимальная глубина: {best_rf_class.max_depth}")

# Улучшенная модель случайного леса для регрессии
best_rf_reg = grid_rf_reg.best_estimator_
y_pred_rf_reg_improved = best_rf_reg.predict(X_test_reg)

metrics_rf_reg_improved = regression_metrics(y_test_reg, y_pred_rf_reg_improved)
print("\nМетрики регрессии (улучшенный случайный лес):")
for key, value in metrics_rf_reg_improved.items():
    print(f"{key}: {value:.4f}")

print(f"\nКоличество деревьев: {best_rf_reg.n_estimators}")
print(f"Максимальная глубина: {best_rf_reg.max_depth}")

Метрики классификации (улучшенный случайный лес):
Accuracy: 0.9230
Precision: 0.9276
Recall: 0.9181
F1-Score: 0.9198
ROC-AUC: 0.9955

Количество деревьев: 100
Максимальная глубина: 10

Метрики регрессии (улучшенный случайный лес):
MSE: 55.6632
MAE: 3.0073
R2: 0.9838

Количество деревьев: 50
Максимальная глубина: 20


## Сравнение с исходным бейзлайном

In [9]:
print("Сравнение классификации (случайный лес):")
print("Метрика | Исходный | Улучшенный")
for key in metrics_rf_class.keys():
    if key in metrics_rf_class_improved:
        print(f"{key:12} | {metrics_rf_class[key]:.4f} | {metrics_rf_class_improved[key]:.4f}")

print("\nСравнение регрессии (случайный лес):")
print("Метрика | Исходный | Улучшенный")
for key in metrics_rf_reg.keys():
    print(f"{key:12} | {metrics_rf_reg[key]:.4f} | {metrics_rf_reg_improved[key]:.4f}")

Сравнение классификации (случайный лес):
Метрика | Исходный | Улучшенный
Accuracy     | 0.9267 | 0.9230
Precision    | 0.9281 | 0.9276
Recall       | 0.9237 | 0.9181
F1-Score     | 0.9250 | 0.9198
ROC-AUC      | 0.9949 | 0.9955

Сравнение регрессии (случайный лес):
Метрика | Исходный | Улучшенный
MSE          | 84.9714 | 55.6632
MAE          | 3.2122 | 3.0073
R2           | 0.9753 | 0.9838


## Самостоятельная реализация случайного леса для классификации

Реализуем алгоритм случайного леса для классификации "с нуля", используя решающие деревья с бутстрэпингом и случайным выбором признаков.

In [ ]:
import numpy as np
from collections import Counter

class RandomForestClassifierCustom:
    """Самостоятельная реализация случайного леса для классификации"""
    
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2, 
                 min_samples_leaf=1, max_features='sqrt', random_state=None):
        """
        Инициализация случайного леса для классификации.
        
        Параметры:
        ----------
        n_estimators : int, default=100
            Количество деревьев в лесу
        max_depth : int or None, default=None
            Максимальная глубина деревьев
        min_samples_split : int, default=2
            Минимальное количество образцов для разделения узла
        min_samples_leaf : int, default=1
            Минимальное количество образцов в листе
        max_features : str or int, default='sqrt'
            Количество признаков для рассмотрения при каждом разделении
            'sqrt' - квадратный корень от общего числа признаков
            'log2' - логарифм по основанию 2 от общего числа признаков
            int - конкретное количество признаков
        random_state : int or None, default=None
            Случайное начальное число для воспроизводимости
        """
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.n_classes_ = None  # Будет установлено в методе fit
        
        if random_state is not None:
            np.random.seed(random_state)
    
    def _bootstrap_sample(self, X, y):
        """Создание бутстрэп выборки"""
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X[indices], y[indices]
    
    def _get_max_features(self, n_features):
        """Определение количества признаков для рассмотрения"""
        if self.max_features == 'sqrt':
            return int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            return int(np.log2(n_features)) + 1
        elif isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        else:
            return n_features
    
    def _build_decision_tree(self, X, y, depth=0):
        """Рекурсивное построение одного дерева решений"""
        n_samples, n_features = X.shape
        
        # Критерии остановки
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            leaf_value = self._most_common_label(y)
            return {'type': 'leaf', 'value': leaf_value}
        
        # Случайный выбор признаков для разделения
        n_selected_features = self._get_max_features(n_features)
        selected_features = np.random.choice(n_features, n_selected_features, replace=False)
        
        best_gain = -1
        best_split = None
        
        # Поиск наилучшего разделения среди выбранных признаков
        for feature_idx in selected_features:
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold
                
                if np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf:
                    continue
                
                y_left = y[left_mask]
                y_right = y[right_mask]
                
                # Вычисление прироста информации (коэффициент Джини)
                gain = self._gini_gain(y, y_left, y_right)
                
                if gain > best_gain:
                    best_gain = gain
                    best_split = {
                        'feature_idx': feature_idx,
                        'threshold': threshold,
                        'gain': gain,
                        'left_mask': left_mask,
                        'right_mask': right_mask
                    }
        
        if best_split is None or best_gain == 0:
            leaf_value = self._most_common_label(y)
            return {'type': 'leaf', 'value': leaf_value}
        
        # Рекурсивное построение левого и правого поддеревьев
        left_subtree = self._build_decision_tree(X[best_split['left_mask']], y[best_split['left_mask']], depth + 1)
        right_subtree = self._build_decision_tree(X[best_split['right_mask']], y[best_split['right_mask']], depth + 1)
        
        return {
            'type': 'node',
            'feature_idx': best_split['feature_idx'],
            'threshold': best_split['threshold'],
            'left': left_subtree,
            'right': right_subtree
        }
    
    def _gini(self, y):
        """Вычисление коэффициента Джини"""
        if len(y) == 0:
            return 0
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)
    
    def _gini_gain(self, y, y_left, y_right):
        """Вычисление прироста информации по Джини"""
        parent_gini = self._gini(y)
        n = len(y)
        n_left, n_right = len(y_left), len(y_right)
        
        if n_left == 0 or n_right == 0:
            return 0
        
        child_gini = (n_left / n) * self._gini(y_left) + (n_right / n) * self._gini(y_right)
        return parent_gini - child_gini
    
    def _most_common_label(self, y):
        """Нахождение наиболее часто встречающегося класса"""
        if len(y) == 0:
            return 0
        counts = np.bincount(y)
        return np.argmax(counts)
    
    def fit(self, X, y):
        """Обучение случайного леса"""
        n_samples, n_features = X.shape
        self.n_classes_ = len(np.unique(y))
        
        for i in range(self.n_estimators):
            # Создание бутстрэп выборки
            X_boot, y_boot = self._bootstrap_sample(X, y)
            
            tree = self._build_decision_tree(X_boot, y_boot)
            self.trees.append(tree)
            
            if (i + 1) % 10 == 0:
                print(f"Построено {i + 1}/{self.n_estimators} деревьев")
        
        print(f"Обучение завершено. Построено {self.n_estimators} деревьев.")
    
    def _traverse_tree(self, x, tree):
        """Обход дерева для предсказания одного образца"""
        if tree['type'] == 'leaf':
            return tree['value']
        
        if x[tree['feature_idx']] <= tree['threshold']:
            return self._traverse_tree(x, tree['left'])
        else:
            return self._traverse_tree(x, tree['right'])
    
    def predict(self, X):
        """Предсказание меток классов (голосование большинством)"""
        n_samples = X.shape[0]
        all_predictions = []
        
        for tree in self.trees:
            tree_predictions = [self._traverse_tree(x, tree) for x in X]
            all_predictions.append(tree_predictions)
        
        all_predictions = np.array(all_predictions).T
        
        final_predictions = []
        for sample_predictions in all_predictions:
            counts = Counter(sample_predictions)
            final_predictions.append(counts.most_common(1)[0][0])
        
        return np.array(final_predictions)
    
    def predict_proba(self, X):
        """Предсказание вероятностей (доля деревьев, проголосовавших за каждый класс)"""
        n_samples = X.shape[0]
        
        if self.n_classes_ is None:
            raise ValueError("Модель должна быть обучена перед вызовом predict_proba")
        
        proba = np.zeros((n_samples, self.n_classes_))
        
        for tree in self.trees:
            tree_predictions = [self._traverse_tree(x, tree) for x in X]
            for i, cls in enumerate(tree_predictions):
                proba[i, cls] += 1
        
        proba = proba / self.n_estimators
        
        return proba

## Самостоятельная реализация случайного леса для регрессии

Реализуем алгоритм случайного леса для регрессии "с нуля", используя решающие деревья с бутстрэпингом и случайным выбором признаков.

In [ ]:
class RandomForestRegressorCustom:
    """Самостоятельная реализация случайного леса для регрессии"""
    
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2, 
                 min_samples_leaf=1, max_features='sqrt', random_state=None):
        """
        Инициализация случайного леса для регрессии.
        
        Параметры:
        ----------
        n_estimators : int, default=100
            Количество деревьев в лесу
        max_depth : int or None, default=None
            Максимальная глубина деревьев
        min_samples_split : int, default=2
            Минимальное количество образцов для разделения узла
        min_samples_leaf : int, default=1
            Минимальное количество образцов в листе
        max_features : str or int, default='sqrt'
            Количество признаков для рассмотрения при каждом разделении
            'sqrt' - квадратный корень от общего числа признаков
            'log2' - логарифм по основанию 2 от общего числа признаков
            int - конкретное количество признаков
        random_state : int or None, default=None
            Случайное начальное число для воспроизводимости
        """
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        
        if random_state is not None:
            np.random.seed(random_state)
    
    def _bootstrap_sample(self, X, y):
        """Создание бутстрэп выборки"""
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X[indices], y[indices]
    
    def _get_max_features(self, n_features):
        """Определение количества признаков для рассмотрения"""
        if self.max_features == 'sqrt':
            return int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            return int(np.log2(n_features)) + 1
        elif isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        else:
            return n_features
    
    def _mse(self, y):
        """Вычисление среднеквадратичной ошибки"""
        if len(y) == 0:
            return 0
        return np.mean((y - np.mean(y)) ** 2)
    
    def _mse_reduction(self, y, y_left, y_right):
        """Вычисление уменьшения MSE"""
        parent_mse = self._mse(y)
        n = len(y)
        n_left, n_right = len(y_left), len(y_right)
        
        if n_left == 0 or n_right == 0:
            return 0
        
        child_mse = (n_left / n) * self._mse(y_left) + (n_right / n) * self._mse(y_right)
        return parent_mse - child_mse
    
    def _build_decision_tree(self, X, y, depth=0):
        """Рекурсивное построение одного дерева решений"""
        n_samples, n_features = X.shape
        
        # Критерии остановки
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            leaf_value = np.mean(y)
            return {'type': 'leaf', 'value': leaf_value}
        
        # Случайный выбор признаков для разделения
        n_selected_features = self._get_max_features(n_features)
        selected_features = np.random.choice(n_features, n_selected_features, replace=False)
        
        best_reduction = -1
        best_split = None
        
        # Поиск наилучшего разделения среди выбранных признаков
        for feature_idx in selected_features:
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold
                
                if np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf:
                    continue
                
                y_left = y[left_mask]
                y_right = y[right_mask]
                
                reduction = self._mse_reduction(y, y_left, y_right)
                
                if reduction > best_reduction:
                    best_reduction = reduction
                    best_split = {
                        'feature_idx': feature_idx,
                        'threshold': threshold,
                        'reduction': reduction,
                        'left_mask': left_mask,
                        'right_mask': right_mask
                    }
        
        if best_split is None or best_reduction == 0:
            leaf_value = np.mean(y)
            return {'type': 'leaf', 'value': leaf_value}
        
        # Рекурсивное построение левого и правого поддеревьев
        left_subtree = self._build_decision_tree(X[best_split['left_mask']], y[best_split['left_mask']], depth + 1)
        right_subtree = self._build_decision_tree(X[best_split['right_mask']], y[best_split['right_mask']], depth + 1)
        
        return {
            'type': 'node',
            'feature_idx': best_split['feature_idx'],
            'threshold': best_split['threshold'],
            'left': left_subtree,
            'right': right_subtree
        }
    
    def fit(self, X, y):
        """Обучение случайного леса"""
        n_samples, n_features = X.shape
        
        for i in range(self.n_estimators):
            # Создание бутстрэп выборки
            X_boot, y_boot = self._bootstrap_sample(X, y)
            
            tree = self._build_decision_tree(X_boot, y_boot)
            self.trees.append(tree)
            
            if (i + 1) % 10 == 0:
                print(f"Построено {i + 1}/{self.n_estimators} деревьев")
        
        print(f"Обучение завершено. Построено {self.n_estimators} деревьев.")
    
    def _traverse_tree(self, x, tree):
        """Обход дерева для предсказания одного образца"""
        if tree['type'] == 'leaf':
            return tree['value']
        
        if x[tree['feature_idx']] <= tree['threshold']:
            return self._traverse_tree(x, tree['left'])
        else:
            return self._traverse_tree(x, tree['right'])
    
    def predict(self, X):
        """Предсказание значений (усреднение предсказаний всех деревьев)"""
        n_samples = X.shape[0]
        all_predictions = []
        
        for tree in self.trees:
            tree_predictions = [self._traverse_tree(x, tree) for x in X]
            all_predictions.append(tree_predictions)
        
        all_predictions = np.array(all_predictions).T
        
        final_predictions = np.mean(all_predictions, axis=1)
        
        return final_predictions

### Обучение и оценка полностью самостоятельной реализации случайного леса для классификации

Используем оптимальные гиперпараметры, найденные при улучшении бейзлайна.

In [ ]:
# Получаем лучшие параметры из GridSearchCV
best_params_rf_class = grid_rf_class.best_params_

print("Обучение полностью самостоятельной реализации случайного леса для классификации...")

# Адаптация параметров под нашу реализацию
n_estimators_custom = 50  # для ускорения обучения
max_depth_custom = best_params_rf_class.get('max_depth', None)
min_samples_split_custom = best_params_rf_class.get('min_samples_split', 2)
min_samples_leaf_custom = best_params_rf_class.get('min_samples_leaf', 1)
max_features_custom = best_params_rf_class.get('max_features', 'sqrt')

rf_custom_class = RandomForestClassifierCustom(
    n_estimators=n_estimators_custom,
    max_depth=max_depth_custom,
    min_samples_split=min_samples_split_custom,
    min_samples_leaf=min_samples_leaf_custom,
    max_features=max_features_custom,
    random_state=42
)

rf_custom_class.fit(X_train_class.values, y_train_encoded)

# Прогнозы
y_pred_rf_custom = rf_custom_class.predict(X_test_class.values)
y_prob_rf_custom = rf_custom_class.predict_proba(X_test_class.values)

# Оценка метрик
metrics_rf_custom_class = classification_metrics(y_test_encoded, y_pred_rf_custom, y_prob_rf_custom)
print("\nМетрики полностью самостоятельной реализации случайного леса для классификации:")
for key, value in metrics_rf_custom_class.items():
    print(f"{key}: {value:.4f}")

Обучение полностью самостоятельной реализации случайного леса для классификации...
Построено 10/50 деревьев
Построено 20/50 деревьев
Построено 30/50 деревьев
Построено 40/50 деревьев
Построено 50/50 деревьев
Обучение завершено. Построено 50 деревьев.

Метрики полностью самостоятельной реализации случайного леса для классификации:
Accuracy: 0.9257
Precision: 0.9283
Recall: 0.9211
F1-Score: 0.9226
ROC-AUC: 0.9950


### Обучение и оценка полностью самостоятельной реализации случайного леса для регрессии

In [ ]:
# Получаем лучшие параметры из GridSearchCV
best_params_rf_reg = grid_rf_reg.best_params_

print("Обучение полностью самостоятельной реализации случайного леса для регрессии...")

# Адаптация параметров под нашу реализацию
n_estimators_custom_reg = 50  # для ускорения обучения
max_depth_custom_reg = best_params_rf_reg.get('max_depth', None)
min_samples_split_custom_reg = best_params_rf_reg.get('min_samples_split', 2)
min_samples_leaf_custom_reg = best_params_rf_reg.get('min_samples_leaf', 1)
max_features_custom_reg = best_params_rf_reg.get('max_features', 'sqrt')

rf_custom_reg = RandomForestRegressorCustom(
    n_estimators=n_estimators_custom_reg,
    max_depth=max_depth_custom_reg,
    min_samples_split=min_samples_split_custom_reg,
    min_samples_leaf=min_samples_leaf_custom_reg,
    max_features=max_features_custom_reg,
    random_state=42
)

rf_custom_reg.fit(X_train_reg.values, y_train_reg.values)

# Прогнозы
y_pred_rf_custom_reg = rf_custom_reg.predict(X_test_reg.values)

# Оценка метрик
metrics_rf_custom_reg = regression_metrics(y_test_reg, y_pred_rf_custom_reg)
print("\nМетрики полностью самостоятельной реализации случайного леса для регрессии:")
for key, value in metrics_rf_custom_reg.items():
    print(f"{key}: {value:.4f}")

Обучение полностью самостоятельной реализации случайного леса для регрессии...
Построено 10/50 деревьев
Построено 20/50 деревьев
Построено 30/50 деревьев
Построено 40/50 деревьев
Построено 50/50 деревьев
Обучение завершено. Построено 50 деревьев.

Метрики полностью самостоятельной реализации случайного леса для регрессии:
MSE: 63.8364
MAE: 3.3793
R2: 0.9814


## Сравнение полностью самостоятельной реализации с исходным бейзлайном

In [16]:
print("Сравнение полностью самостоятельной реализации случайного леса для классификации с исходным бейзлайном:")
print("Метрика | Самостоятельная | Исходный бейзлайн")
for key in metrics_rf_custom_class.keys():
    if key in metrics_rf_class:
        print(f"{key:12} | {metrics_rf_custom_class[key]:.4f} | {metrics_rf_class[key]:.4f}")

print("\nСравнение полностью самостоятельной реализации случайного леса для регрессии с исходным бейзлайном:")
print("Метрика | Самостоятельная | Исходный бейзлайн")
for key in metrics_rf_custom_reg.keys():
    print(f"{key:12} | {metrics_rf_custom_reg[key]:.4f} | {metrics_rf_reg[key]:.4f}")

Сравнение полностью самостоятельной реализации случайного леса для классификации с исходным бейзлайном:
Метрика | Самостоятельная | Исходный бейзлайн
Accuracy     | 0.9257 | 0.9267
Precision    | 0.9283 | 0.9281
Recall       | 0.9211 | 0.9237
F1-Score     | 0.9226 | 0.9250
ROC-AUC      | 0.9950 | 0.9949

Сравнение полностью самостоятельной реализации случайного леса для регрессии с исходным бейзлайном:
Метрика | Самостоятельная | Исходный бейзлайн
MSE          | 63.8364 | 84.9714
MAE          | 3.3793 | 3.2122
R2           | 0.9814 | 0.9753


## Сравнение полностью самостоятельной реализации с улучшенным бейзлайном

In [17]:
print("Сравнение полностью самостоятельной реализации случайного леса для классификации с улучшенным бейзлайном:")
print("Метрика | Самостоятельная | Улучшенный бейзлайн")
for key in metrics_rf_custom_class.keys():
    if key in metrics_rf_class_improved:
        print(f"{key:12} | {metrics_rf_custom_class[key]:.4f} | {metrics_rf_class_improved[key]:.4f}")

print("\nСравнение полностью самостоятельной реализации случайного леса для регрессии с улучшенным бейзлайном:")
print("Метрика | Самостоятельная | Улучшенный бейзлайн")
for key in metrics_rf_custom_reg.keys():
    print(f"{key:12} | {metrics_rf_custom_reg[key]:.4f} | {metrics_rf_reg_improved[key]:.4f}")

Сравнение полностью самостоятельной реализации случайного леса для классификации с улучшенным бейзлайном:
Метрика | Самостоятельная | Улучшенный бейзлайн
Accuracy     | 0.9257 | 0.9230
Precision    | 0.9283 | 0.9276
Recall       | 0.9211 | 0.9181
F1-Score     | 0.9226 | 0.9198
ROC-AUC      | 0.9950 | 0.9955

Сравнение полностью самостоятельной реализации случайного леса для регрессии с улучшенным бейзлайном:
Метрика | Самостоятельная | Улучшенный бейзлайн
MSE          | 63.8364 | 55.6632
MAE          | 3.3793 | 3.0073
R2           | 0.9814 | 0.9838


## Итоговые выводы по лабораторной работе №4

### 1. Выбор данных и метрик

**Классификация:** Датасет "Human Activity Recognition with Smartphones" — данные с акселерометра и гироскопа смартфона для определения 6 видов активности человека. Практическая задача: мониторинг физической активности в приложениях здоровья и фитнеса.

**Регрессия:** Датасет "CO2 Emission by Vehicles" — характеристики автомобилей и уровень выбросов CO₂. Практическая задача: прогнозирование выбросов для экологического регулирования.

**Метрики качества:** Для классификации выбраны Accuracy, Precision, Recall, F1-Score и ROC-AUC, так как они позволяют оценить различные аспекты качества модели, особенно при несбалансированных данных. Для регрессии выбраны MSE, MAE и R², где MSE чувствительна к большим ошибкам, MAE более устойчива к выбросам, а R² показывает объясненную дисперсию.

### 2. Бейзлайн и его улучшение

**Исходный бейзлайн показал следующие результаты:**

**Случайный лес (классификация):** Accuracy = 0.9267, ROC-AUC = 0.9949, количество деревьев = 100, средняя глубина = 18.62

**Случайный лес (регрессия):** R² = 0.9753, MSE = 84.9714, количество деревьев = 100, средняя глубина = 21.22

**Для случайного леса масштабирование признаков не требуется**, так как алгоритм работает с порядковыми отношениями, а не с абсолютными значениями.

**Подбор гиперпараметров на кросс-валидации дал следующие результаты:**

**Случайный лес (классификация):** Лучшие параметры: max_depth=10, max_features='log2', min_samples_leaf=1, min_samples_split=2, n_estimators=100. Accuracy на кросс-валидации составила 0.9266, что практически идентично исходному бейзлайну (0.9267).

**Случайный лес (регрессия):** Лучшие параметры: max_depth=20, max_features='sqrt', min_samples_leaf=1, min_samples_split=2, n_estimators=50. MSE на кросс-валидации составила 56.0677, что на 34.0% лучше исходного бейзлайна.

**Улучшенные модели на тестовой выборке показали следующие результаты:**

**Случайный лес (классификация):** Accuracy = 0.9230 (снижение на 0.40% относительно исходного), ROC-AUC = 0.9955 (незначительное улучшение на 0.06%)

**Случайный лес (регрессия):** R² = 0.9838 (улучшение на 0.87% относительно исходного), MSE = 55.6632 (улучшение на 34.5% относительно исходного)

**Особенность случайного леса:** Как ансамблевый метод, случайный лес показал высокую точность для обеих задач. Интересно, что для классификации подбор гиперпараметров не дал значительного улучшения на тестовой выборке, а для регрессии улучшение было существенным, особенно по метрике MSE.

### 3. Самостоятельная реализация алгоритмов

Реализованы алгоритмы случайного леса для классификации и регрессии "с нуля" с использованием только базового Python и NumPy.

**Включены ключевые особенности случайного леса:**
1) Бутстрэп агрегирование (Bagging) - создание множества бутстрэп выборок
2) Случайный выбор признаков при каждом разделении узла
3) Голосование большинством для классификации
4) Усреднение предсказаний для регрессии
5) Реализация решающих деревьев с ограничениями глубины и минимального количества образцов

**Результаты самостоятельной реализации:**

**Случайный лес (классификация):** Accuracy = 0.9257 (ниже исходного на 0.11%, но выше улучшенного на 0.29%), ROC-AUC = 0.9950

**Случайный лес (регрессия):** R² = 0.9814 (ниже улучшенного на 0.24%, но выше исходного на 0.63%), MSE = 63.8364 (хуже улучшенного на 14.7%, но лучше исходного на 24.9%)

**Особенности реализации:** Для ускорения обучения количество деревьев в самостоятельной реализации было уменьшено до 50 (по сравнению с 100 в sklearn). Несмотря на это, модель показала очень близкие результаты к библиотечной реализации, что подтверждает корректность реализации алгоритма.

### 4. Сравнение всех подходов

**Классификация (случайный лес):**

Исходный бейзлайн: Accuracy = 0.9267

Улучшенный бейзлайн: Accuracy = 0.9230 (-0.40% относительно исходного)

Самостоятельная реализация: Accuracy = 0.9257 (-0.11% относительно исходного, +0.29% относительно улучшенного)

**Регрессия (случайный лес):**

Исходный бейзлайн: R² = 0.9753, MSE = 84.9714

Улучшенный бейзлайн: R² = 0.9838 (+0.87%), MSE = 55.6632 (-34.5%)

Самостоятельная реализация: R² = 0.9814 (+0.63% относительно исходного, -0.24% относительно улучшенного), MSE = 63.8364 (-24.9% относительно исходного, +14.7% относительно улучшенного)

### 5. Ключевые наблюдения и выводы

**Высокая эффективность случайного леса:** Случайный лес показал отличные результаты на обеих задачах. Для классификации активностей человека Accuracy превысил 0.92, а для регрессии R² приблизился к 0.98, что свидетельствует о высокой предсказательной способности модели.

**Различная реакция на настройку гиперпараметров:** Для классификации настройка гиперпараметров не дала значительного улучшения на тестовой выборке, в то время как для регрессии улучшение было существенным. Это может объясняться тем, что исходная модель для классификации уже была близка к оптимальной, либо особенностями данных.

**Важность ограничения глубины деревьев:** Улучшенные модели имели ограниченную глубину (10 для классификации, 20 для регрессии) по сравнению с неограниченными деревьями в исходном бейзлайне. Это предотвращает переобучение и улучшает обобщающую способность, особенно для регрессии.

**Сравнение с одиночными деревьями:** Случайный лес значительно превзошел одиночные решающие деревья из лабораторной работы №3 (Accuracy: 0.9267 vs 0.8558 для классификации, R²: 0.9838 vs 0.9564 для регрессии). Это подтверждает преимущество ансамблевых методов.

**Самостоятельная реализация:** Показала очень хорошие результаты, особенно учитывая уменьшенное количество деревьев (50 вместо 100). Для классификации наша реализация даже немного превзошла улучшенный бейзлайн sklearn. Это свидетельствует о корректной реализации основных принципов случайного леса: бутстрэпинга, случайного выбора признаков и голосования/усреднения.

**Вычислительные аспекты:** Случайный лес требует значительных вычислительных ресурсов, особенно для большого количества деревьев. Наша реализация на чистом Python оказалась значительно медленнее оптимизированной библиотечной реализации, что ожидаемо.

**Практические рекомендации:**
1. Случайный лес является надежным выбором для широкого круга задач, обеспечивая высокую точность без тщательной настройки гиперпараметров.
2. Для задач регрессии настройка гиперпараметров может дать существенное улучшение качества.
3. Ограничение глубины деревьев важно для предотвращения переобучения.
4. Для увеличения скорости работы можно уменьшать количество деревьев, но это может снизить точность.

**Практическая значимость:** Случайный лес является одним из самых популярных и эффективных алгоритмов машинного обучения в промышленных приложениях. Его способность работать с разнообразными типами данных, устойчивость к переобучению и высокая точность делают его инструментом выбора для многих практических задач.

Работа продемонстрировала как теоретические основы случайного леса, так и практические аспекты его применения, подтвердив эффективность ансамблевых методов для повышения точности и устойчивости моделей машинного обучения.